# ttbaranalysis

This notebook mirrors `ttbaranalysis.py`. Adjust the options below, run the argument cell, then run the analysis cells.

In [ ]:
from coffea import util
from coffea.nanoevents import NanoAODSchema, BaseSchema
import coffea.processor as processor

import itertools
import time
from datetime import date
import json
import os
from types import SimpleNamespace

from dask.distributed import Client, performance_report

import warnings
warnings.filterwarnings("ignore")

default_datastets = ['data', 'TTbar', 'QCD']
default_signals = ['RSGluon', 'ZPrime10', 'ZPrime30', 'ZPrimeDM', 'ZPrime1']

from ttbarprocessor import TTbarResProcessor
from python.functions import printTime, makeSaveDirectories

In [ ]:
# Edit these values before running the notebook.

dataset = list(default_datastets)
# dataset = ['data', 'TTbar']

iov = '2024'
signals = False

# Leave empty lists to run all available subsections.
era = []
pt = []
mass = []
# mass = ['1000', '2000']

blind = False
bkgest = None
# bkgest = '2dalphabet'
toptagger = 'deepak8'
redirector = 'root://cmsxrootd.fnal.gov/'
ttagWP = 'medium'
btagger = 'deepcsv'
ht = '1400'
noSyst = False

dask = False
env = 'lpc'
test = False
nocluster = False

print('Configured notebook options. Edit this cell and rerun it to change settings.')

In [ ]:
def build_args():
    selected_datasets = list(dataset)
    if signals:
        selected_datasets = list(default_signals)

    return SimpleNamespace(
        dataset=selected_datasets,
        iov=iov,
        signals=signals,
        era=list(era),
        pt=list(pt),
        mass=list(mass),
        blind=blind,
        bkgest=bkgest,
        toptagger=toptagger,
        redirector=redirector,
        ttagWP=ttagWP,
        btagger=btagger,
        ht=ht,
        noSyst=noSyst,
        dask=dask,
        env=env,
        test=test,
        nocluster=nocluster,
    )


args = build_args()
print('------args------')
for argname, value in vars(args).items():
    print(argname, '=', value)
print('----------------')

In [ ]:
def run_analysis(args):
    tic = time.time()

    savedir = f'outputs/dy/'

    if args.dask and (args.env == 'lpc' or args.env == 'L'):
        from lpcjobqueue import LPCCondorCluster

    samples = args.dataset
    IOV = args.iov
    useDeepAK8 = True if (args.toptagger == 'deepak8') else False
    useDeepCSV = True if (args.btagger == 'deepcsv') else False
    htCut = 1400.0 if (args.ht == '1400') else 950.0
    dask_memory = '4GB'
    chunksize_dask = 100000
    chunksize_futures = 10000
    maxchunks = 10 if args.test else None

    systematics = [
        'nominal',
        'jes',
        'jer',
        'pileup',
        'pdf',
        'q2',
        'ttag_pt1',
        'ttag_pt2',
        'ttag_pt3'
    ]

    if ('2016' in IOV) or ('2017' in IOV):
        systematics.append('prefiring')

    if args.bkgest == '2dalphabet':
        systematics.append('transferFunction')

    systematics = ['nominal', 'jes', 'pileup']
    ttagcats = ['at', '2t']
    ycats = ['cen', 'fwd']

    anacats = [t + y for t, y in itertools.product(ttagcats, ycats)]
    label_map = {i: label for i, label in enumerate(anacats)}

    with open('out.log', 'w') as f:
        print('\n' + date.today().isoformat(), file=f)
        print('\n------args------', file=f)
        for argname, value in vars(args).items():
            print(argname, '=', value, file=f)
        print('----------------\n', file=f)
        print('categories =', label_map, file=f)
        print('\n', file=f)
        if not args.noSyst:
            print('systematics =', systematics, file=f)

    print('\n------args------')
    for argname, value in vars(args).items():
        print(argname, '=', value)
    if not args.noSyst:
        print('systematics =', systematics)
    print('----------------\n')

    if args.env == 'casa' or args.env == 'C':
        redirector = 'root://xcache/'
    elif args.env == 'winterfell' or args.env == 'W':
        redirector = '/mnt/data/cms/'
    else:
        redirector = args.redirector

    jsonfiles = {
        'data': 'data/nanoAOD/data.json',
        'QCD': 'data/nanoAOD/QCD.json',
        'TTbar': 'data/nanoAOD/TTbar.json',
        'ZPrime1': 'data/nanoAOD/ZPrime1.json',
        'ZPrime10': 'data/nanoAOD/ZPrime10.json',
        'ZPrime30': 'data/nanoAOD/ZPrime30.json',
        'ZPrimeDM': 'data/nanoAOD/ZPrimeDM.json',
        'RSGluon': 'data/nanoAOD/RSGluon.json',
    }

    upload_to_dask = ['data', 'python', 'ttbarprocessor.py']

    if not os.path.exists(savedir):
        os.makedirs(savedir)
        os.makedirs(savedir + 'logs/')
        os.makedirs(savedir + 'scale/')
        os.makedirs(savedir + 'twodalphabet/')
        os.popen('cp ttbarprocessor.py ' + savedir + 'logs/ttbarprocessor_' + date.today().isoformat().replace('-', '') + '.py')
        os.popen('cat out.log >> ' + savedir + 'logs/ttbarprocessor_diff.txt')
    else:
        for f in os.listdir(savedir + 'logs/'):
            if 'ttbarprocessor' in f and 'py' in f:
                os.popen('cat out.log >> ' + savedir + 'logs/ttbarprocessor_diff.txt')
                print('diff ttbarprocessor.py ' + savedir + 'logs/' + f + ' >> ' + savedir + 'logs/ttbarprocessor_diff.txt')
                os.popen('diff ttbarprocessor.py ' + savedir + 'logs/' + f + ' >> ' + savedir + 'logs/ttbarprocessor_diff.txt')

        if not os.path.exists(savedir):
            os.makedirs(savedir + 'scale/')
        if not os.path.exists(savedir + 'twodalphabet/'):
            os.makedirs(savedir + 'twodalphabet/')

    makeSaveDirectories(coffea_dir=savedir)

    metrics = None

    for sample in samples:
        skipbadfiles = False
        inputfile = jsonfiles[sample]

        with open(inputfile) as json_file:
            subsections = args.era + args.mass + args.pt
            data = json.load(json_file)

            filedict = {}

            try:
                data[IOV].keys()

                if len(subsections) > 0:
                    for s in subsections:
                        if s in data[IOV].keys():
                            filedict[s] = data[IOV][s]
                        else:
                            print(f'{s} not in {sample} {IOV}')
                else:
                    filedict = data[IOV]
            except Exception:
                filedict[''] = data[IOV]

            for subsection, files in filedict.items():
                files = [redirector + f for f in files]
                if args.test:
                    files = [files[int(len(files) / 2)]]

                fileset = {sample: files}

                print(files[0])

                subString = subsection.replace('700to', '_700to').replace('1000to', '_1000to')
                if args.bkgest:
                    subString += '_bkgest'

                if (args.toptagger == 'cmsv2') and (args.btagger == 'csvv2'):
                    savedir = 'outputs/oldanalysis/'

                savefilename = f'{savedir}{sample}_{IOV}{subString}.coffea'
                if 'RSGluon' in sample:
                    subString = subString.replace(subsection, '')
                    savefilename = f'{savedir}{sample}{subsection}_{IOV}{subString}.coffea'
                elif 'ZPrime' in sample:
                    subString = subString.replace(subsection, '')
                    savefilename = f'{savedir}ZPrime{subsection}_{sample.replace("ZPrime", "")}_{IOV}{subString}.coffea'
                print(f'running {IOV} {sample} {subsection}')

                if args.toptagger == 'cmsv2':
                    savefilename = savefilename.replace('.coffea', '_cmsv2.coffea')
                if args.btagger == 'csvv2':
                    savefilename = savefilename.replace('.coffea', '_csvv2.coffea')
                if args.ht == '950':
                    savefilename = savefilename.replace('.coffea', '_ht950.coffea')
                if args.blind:
                    savefilename = savefilename.replace('.coffea', '_blind.coffea')
                if args.noSyst:
                    savefilename = savefilename.replace('.coffea', '_noSyst.coffea')
                if args.test:
                    savefilename = savefilename.replace('.coffea', '_test.coffea')

                if not args.dask:
                    output, metrics = processor.run_uproot_job(
                        fileset,
                        treename='Events',
                        processor_instance=TTbarResProcessor(
                            iov=IOV,
                            bkgEst=args.bkgest,
                            noSyst=args.noSyst,
                            deepAK8Cut=args.ttagWP,
                            useDeepAK8=useDeepAK8,
                            useDeepCSV=useDeepCSV,
                            htCut=htCut,
                            anacats=anacats,
                            systematics=systematics,
                            blinding=args.blind,
                        ),
                        executor=processor.futures_executor,
                        executor_args={
                            'skipbadfiles': skipbadfiles,
                            'savemetrics': True,
                            'schema': NanoAODSchema,
                            'workers': 4,
                            'xrootdtimeout': 500,
                        },
                        chunksize=chunksize_futures,
                        maxchunks=maxchunks,
                    )
                else:
                    if args.dask and (args.env == 'lpc' or args.env == 'L'):
                        if args.nocluster:
                            cluster = None
                        else:
                            cluster = LPCCondorCluster(memory=dask_memory, transfer_input_files=upload_to_dask, scheduler_options={'dashboard_address': ':8787'})
                            cluster.adapt(minimum=1, maximum=100)
                    else:
                        cluster = None

                    upload_to_dask = [
                        'data',
                        'python',
                        'ttbarprocessor.py',
                    ]

                    with Client(cluster) as client:
                        run_instance = processor.Runner(
                            metadata_cache={},
                            executor=processor.DaskExecutor(client=client, retries=12),
                            schema=NanoAODSchema,
                            savemetrics=True,
                            skipbadfiles=skipbadfiles,
                            chunksize=chunksize_dask,
                            maxchunks=maxchunks,
                        )

                        if args.nocluster:
                            worker_toc = time.time()
                            print('Waiting for 4 workers...')
                            client.wait_for_workers(4)
                            worker_tic = time.time()
                        else:
                            worker_toc = time.time()
                            print('Waiting for at least one worker...')
                            client.wait_for_workers(1)
                            worker_tic = time.time()

                        print(f'time to wait for worker = {int(worker_tic - worker_toc)}s')

                        output, metrics = run_instance(
                            fileset,
                            treename='Events',
                            processor_instance=TTbarResProcessor(
                                iov=IOV,
                                bkgEst=args.bkgest,
                                noSyst=args.noSyst,
                                deepAK8Cut=args.ttagWP,
                                useDeepAK8=useDeepAK8,
                                useDeepCSV=useDeepCSV,
                                htCut=htCut,
                                anacats=anacats,
                                systematics=systematics,
                                blinding=args.blind,
                            ),
                        )

                        client.shutdown()
                        del cluster

                output['analysisCategories'] = label_map
                util.save(output, savefilename)
                print('saving', savefilename)

    elapsed = time.time() - tic
    printTime(elapsed)
    if metrics is not None:
        print(f"Events/s: {metrics['entries'] / elapsed:.0f}")

    return {'elapsed': elapsed, 'metrics': metrics}


In [ ]:
args = build_args()
run_summary = run_analysis(args)